<div dir="rtl">

# NB05-M01 — Translation Ingestion and Acceptance

## هدف
ساخت Artifactهای Verse-aligned برای فولادوند و انصاریان با حفظ متن و Provenance.

خروجی فعلاً Internal Research Artifact است و Product activation ندارد.

</div>

In [1]:
# NB05-C01 — Imports and project root
import json
import sys
from pathlib import Path

current = Path.cwd().resolve()
project_root = next(
    (p for p in [current, *current.parents] if (p / ".git").exists() and (p / "data").exists()),
    None,
)
if project_root is None:
    raise RuntimeError("QRAG project root not found.")

src = project_root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from qrag_phase1.foundation import collect_git_info, write_json_atomic
from qrag_phase1.quran import parse_quran_pipe_file, read_jsonl, sha256_file, write_jsonl_atomic
from qrag_phase1.translation import build_translation_records, validate_translation_alignment

assert collect_git_info(project_root)["branch"] == "qrag-phase1"

In [2]:
# NB05-C02 — Load upstream inventory and inputs
reports = project_root / "reports/phase1"
upstream = json.loads(
    (reports / "translation_inventory_acceptance_v1.json").read_text(encoding="utf-8")
)
assert upstream["acceptance_status"] == "PASS_WITH_LIMITATIONS"
assert upstream["active_translation_count"] == 0

contract = json.loads(
    (project_root / "config/phase1/translation_contract_v1.json").read_text(encoding="utf-8")
)
quran_rows = read_jsonl(
    project_root / contract["canonical_quran_input"]
)
assert len(quran_rows) == 6236

<div dir="rtl">

# NB05-M03 — Build Quarantined Translation Artifacts

ساخت Artifact به معنی Product permission نیست.

</div>

In [3]:
# NB05-C03 — Build translation artifacts
processed_dir = project_root / "data/phase1/processed/translations"
processed_dir.mkdir(parents=True, exist_ok=True)

artifact_rows = {}

for source in contract["sources"]:
    raw_path = project_root / source["raw_file"]
    parsed = parse_quran_pipe_file(raw_path)
    raw_hash = sha256_file(raw_path)

    rows = build_translation_records(
        parsed["records"],
        source["translation_id"],
        source["source_id"],
        raw_hash,
    )
    alignment = validate_translation_alignment(quran_rows, rows)
    assert alignment["status"] == "PASS"

    output_path = processed_dir / f'{source["translation_id"]}_v1.jsonl'
    write_jsonl_atomic(output_path, rows)
    reloaded = read_jsonl(output_path)

    assert reloaded == rows
    artifact_rows[source["translation_id"]] = {
        "path": str(output_path.relative_to(project_root)),
        "sha256": sha256_file(output_path),
        "record_count": len(reloaded),
        "source_id": source["source_id"],
        "rights_status": source["rights_status"],
        "usage_scope": "INTERNAL_RESEARCH_QUARANTINE",
    }

print(json.dumps(artifact_rows, ensure_ascii=False, indent=2))

{
  "fa.fooladvand": {
    "path": "data/phase1/processed/translations/fa.fooladvand_v1.jsonl",
    "sha256": "fc6e0765f06b3c05f047772454e5539e28f3b6a8c4b01c932018ee4eb52228e5",
    "record_count": 6236,
    "source_id": "translation:fa:fooladvand:legacy-repo-v1",
    "rights_status": "evidence_required",
    "usage_scope": "INTERNAL_RESEARCH_QUARANTINE"
  },
  "fa.ansarian": {
    "path": "data/phase1/processed/translations/fa.ansarian_v1.jsonl",
    "sha256": "c14f706183acdf5d1741180565dda578f09a1a187b39779407abff288d002ba8",
    "record_count": 6236,
    "source_id": "translation:fa:ansarian:legacy-repo-v1",
    "rights_status": "evidence_required",
    "usage_scope": "INTERNAL_RESEARCH_QUARANTINE"
  }
}


In [4]:
# NB05-C04 — Cross-translation key parity
fooladvand = read_jsonl(
    project_root / artifact_rows["fa.fooladvand"]["path"]
)
ansarian = read_jsonl(
    project_root / artifact_rows["fa.ansarian"]["path"]
)

fooladvand_keys = [row["verse_id"] for row in fooladvand]
ansarian_keys = [row["verse_id"] for row in ansarian]
quran_keys = [row["verse_id"] for row in quran_rows]

assert fooladvand_keys == quran_keys
assert ansarian_keys == quran_keys
print("Cross-translation verse parity: PASS")

Cross-translation verse parity: PASS


<div dir="rtl">

# NB05-M05 — Manifest and Quarantine Acceptance

Artifactها ساخته می‌شوند، ولی active_translation_count عمداً صفر باقی می‌ماند.

</div>

In [5]:
# NB05-C05 — Persist manifest and acceptance
artifacts_dir = project_root / "artifacts/phase1/translations"
artifacts_dir.mkdir(parents=True, exist_ok=True)

manifest = {
    "schema_version": "qrag_translation_artifact_manifest_v1",
    "release_id": "qrag_phase1_translation_research_v1",
    "artifacts": artifact_rows,
    "active_translation_count": 0,
    "activation_blocker": "SOURCE_SPECIFIC_REPUBLICATION_RIGHTS_NOT_VERIFIED",
    "status": "RESEARCH_ARTIFACTS_QUARANTINED",
}
manifest_path = artifacts_dir / "translation_manifest_v1.json"
write_json_atomic(manifest_path, manifest)

acceptance = {
    "schema_version": "qrag_nb05_acceptance_v1",
    "objective_status": "ACHIEVED",
    "acceptance_status": "PASS_WITH_LIMITATIONS",
    "failed_checks": [],
    "artifact_translation_count": len(artifact_rows),
    "active_translation_count": 0,
    "warnings": [
        "TRANSLATIONS_ARE_QUARANTINED_INTERNAL_RESEARCH_ARTIFACTS",
    ],
    "next_notebook": "notebooks/phase1/06_tafsir_inventory.ipynb",
}
acceptance_path = reports / "translation_ingestion_acceptance_v1.json"
write_json_atomic(acceptance_path, acceptance)

assert json.loads(manifest_path.read_text(encoding="utf-8")) == manifest
assert json.loads(acceptance_path.read_text(encoding="utf-8")) == acceptance
print(json.dumps(acceptance, ensure_ascii=False, indent=2))

{
  "schema_version": "qrag_nb05_acceptance_v1",
  "objective_status": "ACHIEVED",
  "acceptance_status": "PASS_WITH_LIMITATIONS",
  "failed_checks": [],
  "artifact_translation_count": 2,
  "active_translation_count": 0,
  "warnings": [
    "TRANSLATIONS_ARE_QUARANTINED_INTERNAL_RESEARCH_ARTIFACTS"
  ],
  "next_notebook": "notebooks/phase1/06_tafsir_inventory.ipynb"
}


<div dir="rtl">

# NB05-M06 — جمع‌بندی

Translation pipeline از Quran canonical جدا شده، Text fidelity و Alignment را حفظ می‌کند و Rights uncertainty را پنهان نمی‌کند.

</div>